# Linear Regression on Two Datasets + Gradio and Flask UIs

This notebook trains and evaluates **two linear regression models**:

1. **Rainfall.csv** — predicts numeric `temparature` using numeric weather features. The `rainfall` column contains `yes/no`, so it is a classification target and is intentionally **not** used as the regression target.
2. **Diabetes dataset** — scikit-learn's built-in regression dataset, predicting a numeric disease-progression measure from 10 numeric features.

It then creates a **Gradio UI** and a **Flask web form** for predictions.

> Keep `Rainfall.csv` in the same folder as this notebook. If running in Google Colab, upload it when prompted.

## 1. Install required libraries
Run this cell once if the packages are not installed.

In [ ]:
%pip install -q pandas numpy scikit-learn matplotlib gradio flask

## 2. Imports and load the rainfall dataset

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Change this path if Rainfall.csv is stored somewhere else.
RAIN_FILE = "Rainfall.csv"

if not os.path.exists(RAIN_FILE):
    # Helpful for Google Colab; in local Jupyter, place Rainfall.csv beside this notebook.
    try:
        from google.colab import files
        uploaded = files.upload()
        RAIN_FILE = next((name for name in uploaded if name.lower().endswith(".csv")), RAIN_FILE)
    except Exception:
        pass

if not os.path.exists(RAIN_FILE):
    raise FileNotFoundError("Rainfall.csv was not found. Put it beside this notebook or upload it.")

rain_df = pd.read_csv(RAIN_FILE)
rain_df.columns = rain_df.columns.str.strip()
print("Rainfall dataset shape:", rain_df.shape)
display(rain_df.head())
print("\nColumns:", rain_df.columns.tolist())


## 3. Prepare and train the Rainfall dataset regression model
The CSV's `rainfall` field contains `yes/no`, so it is categorical. We use `temparature` as a numeric regression target. Non-numeric columns are excluded, and missing values are imputed.

In [ ]:
rain_target = "temparature"
if rain_target not in rain_df.columns:
    raise ValueError(f"Expected numeric target column '{rain_target}'. Found: {rain_df.columns.tolist()}")

rain_numeric = rain_df.select_dtypes(include=np.number).copy()
rain_numeric = rain_numeric.replace([np.inf, -np.inf], np.nan)
rain_numeric = rain_numeric.dropna(axis=1, how="all")

# Avoid leakage by removing the target from input features.
rain_features = [c for c in rain_numeric.columns if c != rain_target]
X_rain = rain_numeric[rain_features]
y_rain = rain_numeric[rain_target]

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_rain, y_rain, test_size=0.2, random_state=42
)

rain_model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LinearRegression()
)
rain_model.fit(Xr_train, yr_train)
rain_pred = rain_model.predict(Xr_test)

rain_metrics = {
    "MAE": mean_absolute_error(yr_test, rain_pred),
    "RMSE": np.sqrt(mean_squared_error(yr_test, rain_pred)),
    "R2": r2_score(yr_test, rain_pred)
}
print("Rainfall dataset — target:", rain_target)
print("Features:", rain_features)
print(pd.Series(rain_metrics).round(4))


In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(yr_test, rain_pred, alpha=0.75)
lims = [
    min(float(np.min(yr_test)), float(np.min(rain_pred))),
    max(float(np.max(yr_test)), float(np.max(rain_pred)))
]
plt.plot(lims, lims, "--")
plt.xlabel("Actual temperature")
plt.ylabel("Predicted temperature")
plt.title("Rainfall dataset: actual vs predicted")
plt.tight_layout()
plt.show()


## 4. Load and train the second regression dataset (Diabetes)

In [ ]:
diab = load_diabetes(as_frame=True)
diab_df = diab.frame.copy()
diab_target = "target"
diab_features = [c for c in diab_df.columns if c != diab_target]
X_diab = diab_df[diab_features]
y_diab = diab_df[diab_target]

Xd_train, Xd_test, yd_train, yd_test = train_test_split(
    X_diab, y_diab, test_size=0.2, random_state=42
)

diabetes_model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LinearRegression()
)
diabetes_model.fit(Xd_train, yd_train)
diab_pred = diabetes_model.predict(Xd_test)

diabetes_metrics = {
    "MAE": mean_absolute_error(yd_test, diab_pred),
    "RMSE": np.sqrt(mean_squared_error(yd_test, diab_pred)),
    "R2": r2_score(yd_test, diab_pred)
}
print("Diabetes dataset features:", diab_features)
print(pd.Series(diabetes_metrics).round(4))
display(diab_df.head())


In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(yd_test, diab_pred, alpha=0.75)
lims = [
    min(float(np.min(yd_test)), float(np.min(diab_pred))),
    max(float(np.max(yd_test)), float(np.max(diab_pred)))
]
plt.plot(lims, lims, "--")
plt.xlabel("Actual target")
plt.ylabel("Predicted target")
plt.title("Diabetes dataset: actual vs predicted")
plt.tight_layout()
plt.show()


## 5. Prediction helper functions
The UIs below accept one value for each feature. For rainfall, the initial values are taken from a real row in the CSV. For diabetes, feature values use the dataset's feature scale (standardized numeric values).

In [ ]:
# Example defaults from an observed row for the rainfall model
rain_defaults = X_rain.median(numeric_only=True).to_dict()
diab_defaults = X_diab.median(numeric_only=True).to_dict()

def predict_rainfall_temperature(values):
    row = pd.DataFrame([{feature: float(values.get(feature, rain_defaults[feature]))
                         for feature in rain_features}], columns=rain_features)
    prediction = float(rain_model.predict(row)[0])
    return round(prediction, 3)

def predict_diabetes_target(values):
    row = pd.DataFrame([{feature: float(values.get(feature, diab_defaults[feature]))
                         for feature in diab_features}], columns=diab_features)
    prediction = float(diabetes_model.predict(row)[0])
    return round(prediction, 3)

print("Rainfall UI defaults:", rain_defaults)
print("Diabetes UI defaults:", diab_defaults)


## 6. Gradio user interface
Run this cell in its own notebook cell. Gradio will display a local interface and may provide a public share link if `share=True` (optional).

In [ ]:
import gradio as gr

with gr.Blocks(title="Linear Regression Predictor") as demo:
    gr.Markdown("# Linear Regression Predictor")
    gr.Markdown("Choose a model, enter feature values, and get a numeric prediction.")

    with gr.Tab("Rainfall.csv — Temperature"):
        gr.Markdown("Predicts `temparature`. Rainfall yes/no is categorical and is not the regression target.")
        rain_inputs = [
            gr.Number(label=feature, value=float(rain_defaults[feature]))
            for feature in rain_features
        ]
        rain_output = gr.Number(label="Predicted temperature")
        rain_button = gr.Button("Predict temperature")
        rain_button.click(
            fn=lambda *vals: predict_rainfall_temperature(dict(zip(rain_features, vals))),
            inputs=rain_inputs,
            outputs=rain_output
        )

    with gr.Tab("Diabetes dataset"):
        gr.Markdown("Predicts the dataset's numeric target. Inputs should use the dataset's feature scale.")
        diab_inputs = [
            gr.Number(label=feature, value=float(diab_defaults[feature]))
            for feature in diab_features
        ]
        diab_output = gr.Number(label="Predicted target")
        diab_button = gr.Button("Predict target")
        diab_button.click(
            fn=lambda *vals: predict_diabetes_target(dict(zip(diab_features, vals))),
            inputs=diab_inputs,
            outputs=diab_output
        )

demo.launch(share=False)


## 7. Flask user interface
This starts a Flask server on port 5000. The form accepts feature values as JSON so both models can be used without creating a very long HTML form.

- Select the model (`rainfall` or `diabetes`).
- Paste a JSON object containing feature names and numeric values.
- Missing feature keys use median defaults from the training data.

**Jupyter note:** Flask's development server runs in the notebook process. Stop it with the notebook interrupt button. In local Jupyter, open `http://127.0.0.1:5000`. In hosted notebooks such as Colab, you may need a tunnel (for example, ngrok) to access the server from your browser.

In [ ]:
from flask import Flask, request, render_template_string

app = Flask("linear_regression_demo")

HTML_PAGE = '''
<!doctype html>
<html>
<head>
  <title>Linear Regression Predictor</title>
  <style>
    body { font-family: Arial, sans-serif; max-width: 900px; margin: 40px auto; padding: 0 16px; }
    textarea { width: 100%; height: 180px; font-family: monospace; }
    select, button { padding: 8px; margin: 8px 0; }
    .result { padding: 12px; background: #eef7ee; border-radius: 6px; }
    .error { padding: 12px; background: #fdecec; border-radius: 6px; }
  </style>
</head>
<body>
  <h1>Linear Regression Predictor</h1>
  <p>Choose a model and submit numeric features as a JSON object.</p>
  <form method="post">
    <label for="model">Dataset / model:</label><br>
    <select name="model" id="model">
      <option value="rainfall" {% if model == 'rainfall' %}selected{% endif %}>Rainfall.csv — temperature</option>
      <option value="diabetes" {% if model == 'diabetes' %}selected{% endif %}>Diabetes dataset</option>
    </select>
    <p><label for="features">Feature values (JSON):</label></p>
    <textarea name="features" id="features">{{ features_text }}</textarea><br>
    <button type="submit">Predict</button>
  </form>
  {% if result is not none %}<div class="result"><b>Prediction:</b> {{ result }}</div>{% endif %}
  {% if error %}<div class="error"><b>Error:</b> {{ error }}</div>{% endif %}
  <h3>Expected feature names</h3>
  <p><b>Rainfall:</b> {{ rain_feature_names }}</p>
  <p><b>Diabetes:</b> {{ diabetes_feature_names }}</p>
</body>
</html>
'''

@app.route("/", methods=["GET", "POST"])
def home():
    model_name = "rainfall"
    defaults = rain_defaults
    result = None
    error = None
    features_text = json.dumps(rain_defaults, indent=2)

    if request.method == "POST":
        model_name = request.form.get("model", "rainfall")
        features_text = request.form.get("features", "{}")
        defaults = rain_defaults if model_name == "rainfall" else diab_defaults
        try:
            values = json.loads(features_text)
            if not isinstance(values, dict):
                raise ValueError("Input must be a JSON object mapping feature names to numbers.")
            if model_name == "rainfall":
                result = predict_rainfall_temperature(values)
            elif model_name == "diabetes":
                result = predict_diabetes_target(values)
            else:
                raise ValueError("Unknown model selected.")
        except Exception as exc:
            error = str(exc)

    return render_template_string(
        HTML_PAGE,
        model=model_name,
        features_text=features_text,
        result=result,
        error=error,
        rain_feature_names=", ".join(rain_features),
        diabetes_feature_names=", ".join(diab_features)
    )

# use_reloader=False prevents Flask from trying to restart the notebook kernel.
app.run(host="127.0.0.1", port=5000, debug=False, use_reloader=False)


## 8. Notes / interpretation
- **MAE**: average absolute prediction error; lower is better.
- **RMSE**: penalizes larger errors more; lower is better.
- **R²**: proportion of target variation explained by the model; closer to 1 is generally better.
- Linear regression is appropriate when the target is numeric. If your assignment specifically requires predicting the `rainfall` yes/no column, use a classification algorithm such as logistic regression instead.
- The built-in Diabetes dataset is loaded by scikit-learn, so it does not require downloading a second CSV.